### Quadrotor Attack Effect Analysis

This notebook runs two standalone quadrotor scenarios and plots each result independently. The nominal section provides the baseline response, and the acoustic injection section uses the attack scenario with speaker power configured in the YAML file.

In [37]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "scenarios").is_dir() and (path / "src" / "cp_glimpse_py").is_dir():
            return path
    raise RuntimeError("Could not find FIRE_CP_Glimpse repository root.")


REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (11, 7),
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print(REPO_ROOT)

/home/mhcho/ws/FIRE_CP_Glimpse


In [38]:
SCENARIOS = {
    "nominal": REPO_ROOT / "scenarios" / "quadrotor_nominal.yaml",
    "acoustic_attack": REPO_ROOT / "scenarios" / "quadrotor_acoustic_attack.yaml",
    "gps_spoofing": REPO_ROOT / "scenarios" / "quadrotor_gps_spoofing.yaml",
    "parameter_change_attack": REPO_ROOT / "scenarios" / "quadrotor_parameter_change_attack.yaml",
    "gyroscope_offset_attack": REPO_ROOT / "scenarios" / "quadrotor_gyroscope_offset_attack.yaml",
    "barometer_offset_attack": REPO_ROOT / "scenarios" / "quadrotor_barometer_offset_attack.yaml"
}

OUTPUTS = {
    name: RESULTS_DIR / name / "outputs.csv"
    for name in SCENARIOS
}

# Set this to False if you only want to reload existing CSV files.
RUN_SIMULATIONS = True

#### Shared Helpers

These cells locate the repository, define scenario/result paths, run simulations through the CLI, load CSV outputs, and provide reusable plotting helpers.

In [41]:
def run_scenario(name: str) -> Path:
    scenario_path = SCENARIOS[name]
    save_dir = OUTPUTS[name].parent
    save_dir.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    src_path = str(REPO_ROOT / "src")
    env["PYTHONPATH"] = src_path + os.pathsep + env.get("PYTHONPATH", "")

    cmd = [
        sys.executable,
        "-m",
        "cp_glimpse_py.main",
        "--scenario",
        str(scenario_path),
        "--save-dir",
        str(save_dir),
    ]
    completed = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.returncode != 0:
        print(completed.stdout[-4000:])
        print(completed.stderr[-4000:])
        raise RuntimeError(f"{name} simulation failed with exit code {completed.returncode}.")

    print(completed.stdout[-1000:])
    return OUTPUTS[name]


def load_result(name: str) -> pd.DataFrame:
    output = OUTPUTS[name]
    if not output.exists():
        raise FileNotFoundError(f"Run the {name} scenario first: {output}")
    return pd.read_csv(output)


def require_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError("Missing columns: " + ", ".join(missing))
    return columns


def plot_3d_position(df: pd.DataFrame, title: str) -> None:
    cols = require_columns(df, [
        "quadrotor.quad_low.position_w_p_w[1]",
        "quadrotor.quad_low.position_w_p_w[2]",
        "quadrotor.quad_low.position_w_p_w[3]",
    ])
    x, y, z = [df[col] for col in cols]

    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(x, y, -z, linewidth=2)
    ax.scatter(x.iloc[0], y.iloc[0], -z.iloc[0], s=50, label="start")
    ax.scatter(x.iloc[-1], y.iloc[-1], -z.iloc[-1], s=50, label="end")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.set_title(title)
    ax.legend(loc="best")
    plt.show()


def plot_timeseries(
    df: pd.DataFrame,
    columns: list[str],
    title: str,
    ylabel: str = "",
    subplot_titles: list[str] | dict[str, str] | None = None,
    ylabels: list[str] | dict[str, str] | None = None,
) -> None:
    cols = require_columns(df, columns)
    fig, axes = plt.subplots(len(cols), 1, sharex=True, figsize=(11, 2.7 * len(cols)))
    if len(cols) == 1:
        axes = [axes]

    def pick_label(labels, col: str, idx: int, default: str) -> str:
        if labels is None:
            return default
        if isinstance(labels, dict):
            return labels.get(col, default)
        if idx < len(labels):
            return labels[idx]
        return default

    for idx, (ax, col) in enumerate(zip(axes, cols)):
        ax.plot(df["time"], df[col])
        ax.set_title(pick_label(subplot_titles, col, idx, col))
        ax.set_ylabel(pick_label(ylabels, col, idx, ylabel or col.split(".")[-1]))

    axes[-1].set_xlabel("time [s]")
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    plt.show()


#### Nominal Scenario

Runs `quadrotor_nominal.yaml` and loads the generated result CSV from `results/nominal/outputs.csv`. <br>
Use this section to inspect the baseline closed-loop flight behavior. <br>
<br>
Plots the baseline flight path in 3D using the quadrotor world-frame position states. <br>
Plots the position / velocity / attitude / angular rate timeseries.

In [ ]:
if RUN_SIMULATIONS:
    run_scenario("nominal")

nominal = load_result("nominal")
nominal.shape

In [ ]:
plot_3d_position(nominal, "Nominal quadrotor trajectory")

plot_timeseries(
    nominal,
    [
        "quadrotor.quad_low.position_w_p_w[1]",
        "quadrotor.quad_low.position_w_p_w[2]",
        "quadrotor.quad_low.position_w_p_w[3]",
    ],
    "Nominal position history",
    subplot_titles=["x position", "y position", "z position"],
    ylabels=["x [m]", "y [m]", "z [m]"],
)

plot_timeseries(
    nominal,
    [
        "quadrotor.quad_low.velocity_w_p_b[1]",
        "quadrotor.quad_low.velocity_w_p_b[2]",
        "quadrotor.quad_low.velocity_w_p_b[3]",
    ],
    "Nominal velocity history",
    subplot_titles=["u velocity", "v velocity", "w velocity"],
    ylabels=["u [m/s]", "v [m/s]", "w [m/s]"],
)

plot_timeseries(
    nominal,
    [
        "quadrotor.euler_wb_meas[1]",
        "quadrotor.euler_wb_meas[2]",
        "quadrotor.euler_wb_meas[3]",
    ],
    "Nominal Euler attitude history",
    subplot_titles=["roll", "pitch", "yaw"],
    ylabels=["R [rad]", "P [rad]", "Y [rad]"],
)

plot_timeseries(
    nominal,
    [
        "quadrotor.quad_low.rate_wb_b[1]",
        "quadrotor.quad_low.rate_wb_b[2]",
        "quadrotor.quad_low.rate_wb_b[3]",
    ],
    "Nominal body rate history",
    subplot_titles=["roll rate", "pitch rate", "yaw rate"],
    ylabels=["p [rad/s]", "q [rad/s]", "r [rad/s]"],
)

plot_timeseries(
    nominal,
    [
        "controller.pwm_rotor_cmd[1]",
        "controller.pwm_rotor_cmd[2]",
        "controller.pwm_rotor_cmd[3]",
        "controller.pwm_rotor_cmd[4]",
    ],
    "Nominal controller PWM history",
    subplot_titles=["rotor 1 PWM", "rotor 2 PWM", "rotor 3 PWM", "rotor 4 PWM"],
    ylabels=["PWM 1", "PWM 2", "PWM 3", "PWM 4"],
)

#### Parameter change attack scenario

Runs `quadrotor_parameter_change_attack.yaml` and loads the generated result CSV from `results/parameter_change_attack/outputs.csv`. <br>
Use this section to inspect the flight behavior under parameter change attack. <br>
<br>
Plots the parameter change attack scenario flight path in 3D using the quadrotor world-frame position states. <br>
Plots the position / velocity / attitude / angular rate timeseries.

In [ ]:
if RUN_SIMULATIONS:
    run_scenario("parameter_change_attack")

parameter_change_attack = load_result("parameter_change_attack")
parameter_change_attack.shape

In [ ]:
plot_3d_position(parameter_change_attack, "Parameter change attack quadrotor trajectory")

plot_timeseries(
    parameter_change_attack,
    [
        "quadrotor.quad_low.position_w_p_w[1]",
        "quadrotor.quad_low.position_w_p_w[2]",
        "quadrotor.quad_low.position_w_p_w[3]",
    ],
    "Parameter change attack position history",
    subplot_titles=["x position", "y position", "z position"],
    ylabels=["x [m]", "y [m]", "z [m]"],
)

plot_timeseries(
    parameter_change_attack,
    [
        "quadrotor.quad_low.velocity_w_p_b[1]",
        "quadrotor.quad_low.velocity_w_p_b[2]",
        "quadrotor.quad_low.velocity_w_p_b[3]",
    ],
    "Parameter change attack velocity history",
    subplot_titles=["u velocity", "v velocity", "w velocity"],
    ylabels=["u [m/s]", "v [m/s]", "w [m/s]"],
)

plot_timeseries(
    parameter_change_attack,
    [
        "quadrotor.euler_wb_meas[1]",
        "quadrotor.euler_wb_meas[2]",
        "quadrotor.euler_wb_meas[3]",
    ],
    "Parameter change attack Euler attitude history",
    subplot_titles=["roll", "pitch", "yaw"],
    ylabels=["R [rad]", "P [rad]", "Y [rad]"],
)

plot_timeseries(
    parameter_change_attack,
    [
        "quadrotor.quad_low.rate_wb_b[1]",
        "quadrotor.quad_low.rate_wb_b[2]",
        "quadrotor.quad_low.rate_wb_b[3]",
    ],
    "Parameter change attack body rate history",
    subplot_titles=["roll rate", "pitch rate", "yaw rate"],
    ylabels=["p [rad/s]", "q [rad/s]", "r [rad/s]"],
)

plot_timeseries(
    parameter_change_attack,
    [
        "controller.pwm_rotor_cmd[1]",
        "controller.pwm_rotor_cmd[2]",
        "controller.pwm_rotor_cmd[3]",
        "controller.pwm_rotor_cmd[4]",
    ],
    "Parameter change attack controller PWM history",
    subplot_titles=["rotor 1 PWM", "rotor 2 PWM", "rotor 3 PWM", "rotor 4 PWM"],
    ylabels=["PWM 1", "PWM 2", "PWM 3", "PWM 4"],
)

#### Gyroscope offset attack scenario

Runs `quadrotor_gyroscope_offset_attack.yaml` and loads the generated result CSV from `results/gyroscope_offset_attack/outputs.csv`. <br>
Use this section to inspect the flight behavior under gyroscope parameter offset attack. <br>
<br>
Plots the gyroscope offset attack scenario flight path in 3D using the quadrotor world-frame position states. <br>
Plots the position / velocity / attitude / angular rate / controller PWM / attack offset timeseries.

In [ ]:
if RUN_SIMULATIONS:
    run_scenario("gyroscope_offset_attack")

gyroscope_offset_attack = load_result("gyroscope_offset_attack")
gyroscope_offset_attack.shape

In [ ]:
plot_3d_position(gyroscope_offset_attack, "Gyroscope offset attack quadrotor trajectory")

plot_timeseries(
    gyroscope_offset_attack,
    [
        "quadrotor.quad_low.position_w_p_w[1]",
        "quadrotor.quad_low.position_w_p_w[2]",
        "quadrotor.quad_low.position_w_p_w[3]",
    ],
    "Gyroscope offset attack position history",
    subplot_titles=["x position", "y position", "z position"],
    ylabels=["x [m]", "y [m]", "z [m]"],
)

plot_timeseries(
    gyroscope_offset_attack,
    [
        "quadrotor.quad_low.velocity_w_p_b[1]",
        "quadrotor.quad_low.velocity_w_p_b[2]",
        "quadrotor.quad_low.velocity_w_p_b[3]",
    ],
    "Gyroscope offset attack velocity history",
    subplot_titles=["u velocity", "v velocity", "w velocity"],
    ylabels=["u [m/s]", "v [m/s]", "w [m/s]"],
)

plot_timeseries(
    gyroscope_offset_attack,
    [
        "quadrotor.euler_wb_meas[1]",
        "quadrotor.euler_wb_meas[2]",
        "quadrotor.euler_wb_meas[3]",
    ],
    "Gyroscope offset attack Euler attitude history",
    subplot_titles=["roll", "pitch", "yaw"],
    ylabels=["R [rad]", "P [rad]", "Y [rad]"],
)

plot_timeseries(
    gyroscope_offset_attack,
    [
        "quadrotor.quad_low.rate_wb_b[1]",
        "quadrotor.quad_low.rate_wb_b[2]",
        "quadrotor.quad_low.rate_wb_b[3]",
    ],
    "Gyroscope offset attack body rate history",
    subplot_titles=["roll rate", "pitch rate", "yaw rate"],
    ylabels=["p [rad/s]", "q [rad/s]", "r [rad/s]"],
)

plot_timeseries(
    gyroscope_offset_attack,
    [
        "controller.pwm_rotor_cmd[1]",
        "controller.pwm_rotor_cmd[2]",
        "controller.pwm_rotor_cmd[3]",
        "controller.pwm_rotor_cmd[4]",
    ],
    "Gyroscope offset attack controller PWM history",
    subplot_titles=["rotor 1 PWM", "rotor 2 PWM", "rotor 3 PWM", "rotor 4 PWM"],
    ylabels=["PWM 1", "PWM 2", "PWM 3", "PWM 4"],
)

plot_timeseries(
    gyroscope_offset_attack,
    [
        "controller.gyro_rate_offset[1]",
        "controller.gyro_rate_offset[2]",
        "controller.gyro_rate_offset[3]",
    ],
    "Gyroscope offset attack parameter history",
    subplot_titles=["p gyro offset", "q gyro offset", "r gyro offset"],
    ylabels=["p offset [rad/s]", "q offset [rad/s]", "r offset [rad/s]"],
)


#### Barometer offset attack scenario

Runs `quadrotor_barometer_offset_attack.yaml` and loads the generated result CSV from `results/barometer_offset_attack/outputs.csv`. <br>
Use this section to inspect the flight behavior under barometer altitude offset attack. <br>
<br>
Plots the barometer offset attack scenario flight path in 3D using the quadrotor world-frame position states. <br>
Plots the position / velocity / attitude / angular rate / controller PWM / altitude offset timeseries.

In [ ]:
if RUN_SIMULATIONS:
    run_scenario("barometer_offset_attack")

barometer_offset_attack = load_result("barometer_offset_attack")
barometer_offset_attack.shape

In [ ]:
plot_3d_position(barometer_offset_attack, "Barometer offset attack quadrotor trajectory")

plot_timeseries(
    barometer_offset_attack,
    [
        "quadrotor.quad_low.position_w_p_w[1]",
        "quadrotor.quad_low.position_w_p_w[2]",
        "quadrotor.quad_low.position_w_p_w[3]",
    ],
    "Barometer offset attack position history",
    subplot_titles=["x position", "y position", "z position"],
    ylabels=["x [m]", "y [m]", "z [m]"],
)

plot_timeseries(
    barometer_offset_attack,
    [
        "quadrotor.quad_low.velocity_w_p_b[1]",
        "quadrotor.quad_low.velocity_w_p_b[2]",
        "quadrotor.quad_low.velocity_w_p_b[3]",
    ],
    "Barometer offset attack velocity history",
    subplot_titles=["u velocity", "v velocity", "w velocity"],
    ylabels=["u [m/s]", "v [m/s]", "w [m/s]"],
)

plot_timeseries(
    barometer_offset_attack,
    [
        "quadrotor.euler_wb_meas[1]",
        "quadrotor.euler_wb_meas[2]",
        "quadrotor.euler_wb_meas[3]",
    ],
    "Barometer offset attack Euler attitude history",
    subplot_titles=["roll", "pitch", "yaw"],
    ylabels=["R [rad]", "P [rad]", "Y [rad]"],
)

plot_timeseries(
    barometer_offset_attack,
    [
        "quadrotor.quad_low.rate_wb_b[1]",
        "quadrotor.quad_low.rate_wb_b[2]",
        "quadrotor.quad_low.rate_wb_b[3]",
    ],
    "Barometer offset attack body rate history",
    subplot_titles=["roll rate", "pitch rate", "yaw rate"],
    ylabels=["p [rad/s]", "q [rad/s]", "r [rad/s]"],
)

plot_timeseries(
    barometer_offset_attack,
    [
        "controller.pwm_rotor_cmd[1]",
        "controller.pwm_rotor_cmd[2]",
        "controller.pwm_rotor_cmd[3]",
        "controller.pwm_rotor_cmd[4]",
    ],
    "Barometer offset attack controller PWM history",
    subplot_titles=["rotor 1 PWM", "rotor 2 PWM", "rotor 3 PWM", "rotor 4 PWM"],
    ylabels=["PWM 1", "PWM 2", "PWM 3", "PWM 4"],
)

plot_timeseries(
    barometer_offset_attack,
    [
        "quadrotor.pos_sensor_offset[1]",
        "quadrotor.pos_sensor_offset[2]",
        "quadrotor.pos_sensor_offset[3]",
    ],
    "Barometer offset attack sensor offset history",
    subplot_titles=["x sensor offset", "y sensor offset", "z/barometer offset"],
    ylabels=["x offset [m]", "y offset [m]", "z offset [m]"],
)


#### Acoustic injection attack scenario

Runs `quadrotor_acoustic_attack.yaml` and loads the generated result CSV from `results/acoustic_attack/outputs.csv`. <br>
Use this section to inspect the flight behavior under acoustic noise injection attack. <br>
<br>
Plots the acoustic injection attack scenario flight path in 3D using the quadrotor world-frame position states. <br>
Plots the position / velocity / attitude / angular rate timeseries.

In [ ]:
if RUN_SIMULATIONS:
    run_scenario("acoustic_attack")

acoustic_attack = load_result("acoustic_attack")
acoustic_attack.shape

In [ ]:
plot_3d_position(acoustic_attack, "Acoustic injection attack quadrotor trajectory")

plot_timeseries(
    acoustic_attack,
    [
        "quadrotor.quad_low.position_w_p_w[1]",
        "quadrotor.quad_low.position_w_p_w[2]",
        "quadrotor.quad_low.position_w_p_w[3]",
    ],
    "Acoustic injection attack position history",
    subplot_titles=["x position", "y position", "z position"],
    ylabels=["x [m]", "y [m]", "z [m]"],
)

plot_timeseries(
    acoustic_attack,
    [
        "quadrotor.quad_low.velocity_w_p_b[1]",
        "quadrotor.quad_low.velocity_w_p_b[2]",
        "quadrotor.quad_low.velocity_w_p_b[3]",
    ],
    "Acoustic injection attack velocity history",
    subplot_titles=["u velocity", "v velocity", "w velocity"],
    ylabels=["u [m/s]", "v [m/s]", "w [m/s]"],
)

plot_timeseries(
    acoustic_attack,
    [
        "quadrotor.euler_wb_meas[1]",
        "quadrotor.euler_wb_meas[2]",
        "quadrotor.euler_wb_meas[3]",
    ],
    "Acoustic injection attack Euler attitude history",
    subplot_titles=["roll", "pitch", "yaw"],
    ylabels=["R [rad]", "P [rad]", "Y [rad]"],
)

plot_timeseries(
    acoustic_attack,
    [
        "quadrotor.quad_low.rate_wb_b[1]",
        "quadrotor.quad_low.rate_wb_b[2]",
        "quadrotor.quad_low.rate_wb_b[3]",
    ],
    "Acoustic injection attack body rate history",
    subplot_titles=["roll rate", "pitch rate", "yaw rate"],
    ylabels=["p [rad/s]", "q [rad/s]", "r [rad/s]"],
)

plot_timeseries(
    acoustic_attack,
    [
        "controller.pwm_rotor_cmd[1]",
        "controller.pwm_rotor_cmd[2]",
        "controller.pwm_rotor_cmd[3]",
        "controller.pwm_rotor_cmd[4]",
    ],
    "Acoustic injection attack controller PWM history",
    subplot_titles=["rotor 1 PWM", "rotor 2 PWM", "rotor 3 PWM", "rotor 4 PWM"],
    ylabels=["PWM 1", "PWM 2", "PWM 3", "PWM 4"],
)

#### GPS spoofing attack scenario

Runs `quadrotor_gps_spoofing.yaml` and loads the generated result CSV from `results/gps_spoofing/outputs.csv`. <br>
Use this section to inspect the flight behavior under gps spoofing attack. <br>
<br>
Plots the gps spoofing attack scenario flight path in 3D using the quadrotor world-frame position states. <br>
Plots the position / velocity / attitude / angular rate timeseries.

In [ ]:
if RUN_SIMULATIONS:
    run_scenario("gps_spoofing")

gps_spoofing_attack = load_result("gps_spoofing")
gps_spoofing_attack.shape

In [ ]:
plot_3d_position(gps_spoofing_attack, "GPS spoofing attack quadrotor trajectory")

plot_timeseries(
    gps_spoofing_attack,
    [
        "quadrotor.quad_low.position_w_p_w[1]",
        "quadrotor.quad_low.position_w_p_w[2]",
        "quadrotor.quad_low.position_w_p_w[3]",
    ],
    "GPS spoofing attack position history",
    subplot_titles=["x position", "y position", "z position"],
    ylabels=["x [m]", "y [m]", "z [m]"],
)

plot_timeseries(
    gps_spoofing_attack,
    [
        "quadrotor.quad_low.velocity_w_p_b[1]",
        "quadrotor.quad_low.velocity_w_p_b[2]",
        "quadrotor.quad_low.velocity_w_p_b[3]",
    ],
    "GPS spoofing attack velocity history",
    subplot_titles=["u velocity", "v velocity", "w velocity"],
    ylabels=["u [m/s]", "v [m/s]", "w [m/s]"],
)

plot_timeseries(
    gps_spoofing_attack,
    [
        "quadrotor.euler_wb_meas[1]",
        "quadrotor.euler_wb_meas[2]",
        "quadrotor.euler_wb_meas[3]",
    ],
    "GPS spoofing attack Euler attitude history",
    subplot_titles=["roll", "pitch", "yaw"],
    ylabels=["R [rad]", "P [rad]", "Y [rad]"],
)

plot_timeseries(
    gps_spoofing_attack,
    [
        "quadrotor.quad_low.rate_wb_b[1]",
        "quadrotor.quad_low.rate_wb_b[2]",
        "quadrotor.quad_low.rate_wb_b[3]",
    ],
    "GPS spoofing attack body rate history",
    subplot_titles=["roll rate", "pitch rate", "yaw rate"],
    ylabels=["p [rad/s]", "q [rad/s]", "r [rad/s]"],
)

plot_timeseries(
    gps_spoofing_attack,
    [
        "controller.pwm_rotor_cmd[1]",
        "controller.pwm_rotor_cmd[2]",
        "controller.pwm_rotor_cmd[3]",
        "controller.pwm_rotor_cmd[4]",
    ],
    "GPS spoofing attack controller PWM history",
    subplot_titles=["rotor 1 PWM", "rotor 2 PWM", "rotor 3 PWM", "rotor 4 PWM"],
    ylabels=["PWM 1", "PWM 2", "PWM 3", "PWM 4"],
)